In [16]:
import cv2
import numpy as np
import os

# -----------------------------
# Logistic Map Generator
# -----------------------------
def logistic_map(x0, r, size):

    sequence = np.zeros(size)
    sequence[0] = x0

    for i in range(1, size):
        sequence[i] = r * sequence[i-1] * (1 - sequence[i-1])

    return sequence


# -----------------------------
# Image Encryption Function
# -----------------------------
def encrypt_image(image, x0=0.54321, r=3.99):

    rows, cols = image.shape
    total_pixels = rows * cols

    chaotic_seq = logistic_map(x0, r, total_pixels + 100)[100:]

    flat_img = image.flatten()

    # Permutation
    perm_indices = np.argsort(chaotic_seq)
    permuted = flat_img[perm_indices]

    # Key stream
    key_stream = np.uint8(np.mod(np.floor(chaotic_seq * 1e14), 256))

    cipher = np.zeros_like(permuted)

    # Forward diffusion
    cipher[0] = permuted[0] ^ key_stream[0]

    for i in range(1, total_pixels):
        cipher[i] = permuted[i] ^ key_stream[i] ^ cipher[i-1]

    encrypted_image = cipher.reshape(rows, cols)

    return encrypted_image


# -----------------------------
# NPCR and UACI Calculation
# -----------------------------
def calculate_npcr_uaci(img1, img2):

    rows, cols = img1.shape
    total_pixels = rows * cols

    diff = np.where(img1 != img2, 1, 0)

    npcr = np.sum(diff) / total_pixels * 100

    abs_diff = np.abs(img1.astype(float) - img2.astype(float))

    uaci = np.sum(abs_diff) / (255 * total_pixels) * 100

    return npcr, uaci


# -----------------------------
# Main Program
# -----------------------------
if __name__ == "__main__":

    # Load image
    image_path = r"C:\Users\me\test-practice-others\image.jpeg"
    
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
    else:
        image = cv2.imread(image_path)
        
        if image is None:
            print(f"Error: Could not read image from {image_path}")
        else:
            # Resize for standard testing
            image = cv2.resize(image, (256,256))

            # Convert to grayscale
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # Encrypt original image
            cipher1 = encrypt_image(gray)

            # Modify one pixel
            modified = gray.copy()
            modified[0,0] = np.uint8((int(modified[0,0]) + 1) % 256)

            # Encrypt modified image
            cipher2 = encrypt_image(modified)

            # Calculate NPCR and UACI
            npcr, uaci = calculate_npcr_uaci(cipher1, cipher2)

            print("Encryption Results")
            print("-------------------")
            print(f"NPCR: {npcr:.2f}%")
            print(f"UACI: {uaci:.2f}%")

            # Save images
            cv2.imwrite("original.png", gray)
            cv2.imwrite("encrypted1.png", cipher1)
            cv2.imwrite("encrypted2.png", cipher2)

            print("Images saved successfully.")

Encryption Results
-------------------
NPCR: 86.58%
UACI: 0.34%
Images saved successfully.


In [19]:
import cv2
import numpy as np
import os

# -----------------------------
# Logistic Map Generator
# -----------------------------
def logistic_map(x0, r, size):

    seq = np.zeros(size)
    seq[0] = x0

    for i in range(1, size):
        seq[i] = r * seq[i-1] * (1 - seq[i-1])

    return seq


# -----------------------------
# Encryption Function
# -----------------------------
def encrypt_image(img, x0=0.5678, r=3.99):

    rows, cols = img.shape
    total = rows * cols

    chaotic_seq = logistic_map(x0, r, total + 100)[100:]

    flat = img.flatten()

    # Permutation
    perm_index = np.argsort(chaotic_seq)
    scrambled = flat[perm_index]

    # Key stream
    key = np.uint8(np.mod(np.floor(chaotic_seq * 1e14), 256))

    # -----------------------------
    # PASS 1 : Forward diffusion
    # -----------------------------
    c1 = np.zeros_like(scrambled)

    iv1 = key[-1]

    c1[0] = scrambled[0] ^ key[0] ^ iv1

    for i in range(1, total):
        c1[i] = scrambled[i] ^ key[i] ^ c1[i-1]

    # -----------------------------
    # PASS 2 : Backward diffusion
    # -----------------------------
    c2 = np.zeros_like(c1)

    iv2 = key[0]

    c2[-1] = c1[-1] ^ key[-1] ^ iv2

    for i in range(total-2, -1, -1):
        c2[i] = c1[i] ^ key[i] ^ c2[i+1]

    return c2.reshape(rows, cols)


# -----------------------------
# NPCR + UACI
# -----------------------------
def calculate_npcr_uaci(c1, c2):

    rows, cols = c1.shape
    total = rows * cols

    diff = np.where(c1 != c2, 1, 0)
    npcr = np.sum(diff) / total * 100

    abs_diff = np.abs(c1.astype(float) - c2.astype(float))
    uaci = np.sum(abs_diff) / (255 * total) * 100

    return npcr, uaci


# -----------------------------
# MAIN PROGRAM
# -----------------------------
if __name__ == "__main__":

    image_path = r"C:\Users\me\test-practice-others\image.jpeg"
    
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
    else:
        img = cv2.imread(image_path)
        
        if img is None:
            print(f"Error: Could not read image from {image_path}")
        else:
            img = cv2.resize(img, (256,256))

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # encrypt original
            cipher1 = encrypt_image(gray)

            # modify one pixel
            modified = gray.copy()
            modified[0,0] = np.uint8((int(modified[0,0]) + 1) % 256)

            # encrypt modified
            cipher2 = encrypt_image(modified)

            # calculate metrics
            npcr, uaci = calculate_npcr_uaci(cipher1, cipher2)

            print("Results")
            print("NPCR =", npcr, "%")
            print("UACI =", uaci, "%")

            cv2.imwrite("original.png", gray)
            cv2.imwrite("encrypted1.png", cipher1)
            cv2.imwrite("encrypted2.png", cipher2)

Results
NPCR = 34.503173828125 %
UACI = 0.13530656403186275 %


In [21]:
import cv2
import numpy as np
import os

# AES-like S-box for better substitution
SBOX = np.array([
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5e, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xd7, 0x4b, 0x55, 0xcf, 0x34, 0xc5, 0x84,
    0xcb, 0xc6, 0x2f, 0x40, 0x58, 0xcf, 0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9,
    0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8, 0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6,
    0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2, 0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7
], dtype=np.uint8)

# Enhanced Logistic Map
def logistic_map(x0, r, size):
    seq = np.zeros(size)
    x = x0
    for i in range(size):
        x = r * x * (1 - x)
        seq[i] = x
    return seq

# Tent Map
def tent_map(x0, size):
    seq = np.zeros(size)
    x = x0
    for i in range(size):
        if x < 0.5:
            x = 2 * x
        else:
            x = 2 * (1 - x)
        seq[i] = x
    return seq

# Advanced Multi-Round Encryption
def encrypt_image(img, x0=0.123456, r=3.9999, rounds=4):
    rows, cols = img.shape
    total = rows * cols
    
    # Generate chaotic sequences
    chaotic1 = logistic_map(x0, r, total + 100)[100:]
    chaotic2 = tent_map(0.456789, total + 100)[100:]
    
    encrypted = img.flatten().astype(np.uint16)
    
    for round_num in range(rounds):
        # --- PERMUTATION PHASE ---
        perm_key = (chaotic1 * 1e10 + chaotic2 * 1e8) % (total)
        perm_indices = np.argsort(perm_key)
        encrypted = encrypted[perm_indices]
        
        # --- SUBSTITUTION PHASE ---
        encrypted_byte = np.uint8(encrypted % 256)
        encrypted_byte = SBOX[encrypted_byte]
        encrypted = encrypted_byte.astype(np.uint16)
        
        # --- DIFFUSION PHASE 1 (Forward) ---
        key1 = np.uint8(np.mod(np.floor(chaotic1 * 1e14), 256))
        key2 = np.uint8(np.mod(np.floor(chaotic2 * 1e14), 256))
        
        encrypted[0] = (encrypted[0] ^ key1[0] ^ key2[0]) % 256
        for i in range(1, total):
            encrypted[i] = (encrypted[i] ^ key1[i] ^ key2[i] ^ encrypted[i-1]) % 256
        
        # --- DIFFUSION PHASE 2 (Backward) ---
        encrypted[-1] = (encrypted[-1] ^ key1[-1]) % 256
        for i in range(total - 2, -1, -1):
            encrypted[i] = (encrypted[i] ^ key2[i] ^ encrypted[i+1]) % 256
        
        # --- 2D SHUFFLING (Rows and Columns) ---
        encrypted_2d = encrypted.reshape(rows, cols)
        
        # Row shift
        for i in range(rows):
            shift = (int(chaotic1[i] * 100)) % cols
            encrypted_2d[i] = np.roll(encrypted_2d[i], shift)
        
        # Column shift
        for j in range(cols):
            shift = (int(chaotic2[j] * 100)) % rows
            encrypted_2d[:, j] = np.roll(encrypted_2d[:, j], shift)
        
        encrypted = encrypted_2d.flatten().astype(np.uint16)
    
    return np.uint8(encrypted % 256).reshape(rows, cols)

# NPCR + UACI Calculation
def calculate_npcr_uaci(c1, c2):
    rows, cols = c1.shape
    total = rows * cols
    
    diff = np.where(c1 != c2, 1, 0)
    npcr = np.sum(diff) / total * 100
    
    abs_diff = np.abs(c1.astype(float) - c2.astype(float))
    uaci = np.sum(abs_diff) / (255 * total) * 100
    
    return npcr, uaci

# MAIN PROGRAM
if __name__ == "__main__":

    image_path = r"C:\Users\me\test-practice-others\image.jpeg"
    
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
    else:
        img = cv2.imread(image_path)
        
        if img is None:
            print(f"Error: Could not read image from {image_path}")
        else:
            img = cv2.resize(img, (256, 256))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Encrypt original
            cipher1 = encrypt_image(gray, x0=0.123456, r=3.9999, rounds=4)
            
            # Modify one pixel
            modified = gray.copy()
            modified[0, 0] = np.uint8((int(modified[0, 0]) + 1) % 256)
            
            # Encrypt modified
            cipher2 = encrypt_image(modified, x0=0.123456, r=3.9999, rounds=4)
            
            # Calculate metrics
            npcr, uaci = calculate_npcr_uaci(cipher1, cipher2)
            
            print("=" * 50)
            print("ADVANCED ENCRYPTION RESULTS")
            print("=" * 50)
            print(f"NPCR: {npcr:.2f}%")
            print(f"UACI: {uaci:.2f}%")
            print(f"Method: 4-Round Multi-Chaotic Encryption")
            print(f"Features: Logistic + Tent map, S-box, 2D shuffle")
            print("=" * 50)
            
            cv2.imwrite("original.png", gray)
            cv2.imwrite("encrypted1.png", cipher1)
            cv2.imwrite("encrypted2.png", cipher2)
            print("Images saved successfully.")

ADVANCED ENCRYPTION RESULTS
NPCR: 99.57%
UACI: 33.40%
Method: 4-Round Multi-Chaotic Encryption
Features: Logistic + Tent map, S-box, 2D shuffle
Images saved successfully.


In [23]:
import cv2
import numpy as np
import os

def generate_logistic_key(x0, r, size):
    sequence = np.zeros(size)
    sequence[0] = x0
    for i in range(1, size):
        sequence[i] = r * sequence[i-1] * (1 - sequence[i-1])
    return sequence

def encrypt_image_v4(img_array, x0_base, r):
    rows, cols = img_array.shape
    total_pixels = rows * cols

    # Plaintext-Dependent Key
    image_sum = np.sum(img_array)
    x_dynamic = (x0_base + (image_sum / (255.0 * total_pixels))) % 1.0
    if x_dynamic <= 0: 
        x_dynamic = 0.5

    # Generating Key & Permutation
    chaotic_seq = generate_logistic_key(x_dynamic, r, total_pixels + 100)[100:]
    flat_img = img_array.flatten()
    sort_indices = np.argsort(chaotic_seq)
    scrambled_img = flat_img[sort_indices]
    key_stream = np.uint8(np.mod(np.floor(chaotic_seq * (10**14)), 256))

    # Pass 1: Forward Chaining
    c1 = np.zeros_like(scrambled_img)
    iv1 = key_stream[-1]
    c1[0] = scrambled_img[0] ^ key_stream[0] ^ iv1
    for i in range(1, total_pixels):
        c1[i] = scrambled_img[i] ^ key_stream[i] ^ c1[i-1]

    # Pass 2: Backward Chaining
    c2 = np.zeros_like(c1)
    iv2 = key_stream[0]
    c2[-1] = c1[-1] ^ key_stream[-1] ^ iv2
    for i in range(total_pixels - 2, -1, -1):
        c2[i] = c1[i] ^ key_stream[i] ^ c2[i+1]

    return c2.reshape(rows, cols)

def calculate_npcr_uaci(c1, c2):
    w, h = c1.shape
    total_pixels = w * h

    diff = np.where(c1 != c2, 1, 0)
    npcr = (np.sum(diff) / total_pixels) * 100

    c1_float = c1.astype(np.float32)
    c2_float = c2.astype(np.float32)
    abs_diff = np.abs(c1_float - c2_float)
    uaci = (np.sum(abs_diff) / (255 * total_pixels)) * 100

    return npcr, uaci

# MAIN PROGRAM
if __name__ == "__main__":

    image_path = r"C:\Users\me\test-practice-others\DSC 17\image.jpeg"
    
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
    else:
        img = cv2.imread(image_path)
        
        if img is None:
            print(f"Error: Could not read image from {image_path}")
        else:
            # Resize and convert to grayscale
            img = cv2.resize(img, (256, 256))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Encrypt original image
            x0_base = 0.3456
            r = 3.99
            cipher1 = encrypt_image_v4(gray, x0_base, r)
            
            # Modify one pixel
            modified = gray.copy()
            modified[0, 0] = np.uint8((int(modified[0, 0]) + 1) % 256)
            
            # Encrypt modified image
            cipher2 = encrypt_image_v4(modified, x0_base, r)
            
            # Calculate NPCR and UACI
            npcr, uaci = calculate_npcr_uaci(cipher1, cipher2)
            
            print("=" * 50)
            print("PLAINTEXT-DEPENDENT ENCRYPTION RESULTS")
            print("=" * 50)
            print(f"NPCR: {npcr:.2f}%")
            print(f"UACI: {uaci:.2f}%")
            print(f"Method: Plaintext-Dependent Key + Dual-Pass Chaining")
            print("=" * 50)
            
            # Save images
            cv2.imwrite("original.png", gray)
            cv2.imwrite("encrypted1.png", cipher1)
            cv2.imwrite("encrypted2.png", cipher2)
            print("Images saved successfully.")

PLAINTEXT-DEPENDENT ENCRYPTION RESULTS
NPCR: 99.65%
UACI: 33.42%
Method: Plaintext-Dependent Key + Dual-Pass Chaining
Images saved successfully.
